# Этап 6. Поиск похожего блюда на замену (kNN)

**Цель этапа:** для блюда, которому классификатор сказал "нужна замена" - найти наиболее похожее блюдо той же категории, которое лучше подходит под цель пользователя.

**Логика работы:**
1. Пользователь вводит блюдо + цель
2. Проверяем, нужна ли замена (через target-метку из разметки, на 100г)
3. Если нужна — kNN ищет ближайших соседей среди блюд той же категории
4. Из соседей выбирается ближайший, у которого target=0 (замена не нужна)

**Почему kNN:** алгоритм k ближайших соседей находит блюда с наиболее похожим
нутриционным профилем в пространстве 7 признаков.

## 1. Загрузка данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/russian_food_labeled.csv')

targets = ['target_weightloss', 'target_gainmass', 'target_balance', 'target_sugar']
labels  = ['Похудение', 'Набор массы', 'Баланс/ЗОЖ', 'Контроль сахара']
label_to_target = dict(zip(labels, targets))

print('Данные загружены успешно')
print('Блюд в датасете:', len(df))
print('Категорий:', df['Category'].nunique())
print('Категории:', sorted(df['Category'].unique()))

Данные загружены успешно
Блюд в датасете: 400
Категорий: 17
Категории: ['Бобовые', 'Выпечка', 'Гарнир', 'Десерт', 'Европейское', 'Завтрак', 'Крупа', 'Молочное', 'Мясо', 'Напиток', 'Овощ', 'Орехи', 'Рыба', 'Салат', 'Суп', 'Фрукт', 'Яйца']


## 2. Нутриционные признаки для kNN

In [2]:

nutrition_features = [
    'Calories (kcal)',
    'Protein (g)',
    'Carbohydrates (g)',
    'Fat (g)',
    'Fiber (g)',
    'Sugars (g)',
    'Sodium (mg)'
]

print('Признаки для kNN (7 нутриционных характеристик):')
for f in nutrition_features:
    print(f'  - {f}')

Признаки для kNN (7 нутриционных характеристик):
  - Calories (kcal)
  - Protein (g)
  - Carbohydrates (g)
  - Fat (g)
  - Fiber (g)
  - Sugars (g)
  - Sodium (mg)


## 4. Основная функция поиска замены

In [3]:
def find_replacement(food_item, goal, k=10):
    """
    Находит замену для блюда под заданную цель пользователя.
    Все значения КБЖУ — на 100г, как хранятся в датасете.

    Параметры:
        food_item: название блюда
        goal: цель пользователя (Похудение / Набор массы / Баланс/ЗОЖ / Контроль сахара)
        k: не используется как жёсткое ограничение — оставлен для совместимости сигнатуры

    Возвращает словарь с результатом и рекомендацией.
    """
    target_col = label_to_target[goal]

    dish_rows = df[df['Food_Item'] == food_item]
    if len(dish_rows) == 0:
        return {'error': f'Блюдо "{food_item}" не найдено в датасете'}

    dish = dish_rows.iloc[0]
    category = dish['Category']
    needs_replacement = int(dish[target_col]) == 1
    original = {f: float(dish[f]) for f in nutrition_features}

    if not needs_replacement:
        return {
            'food_item': food_item, 'goal': goal, 'category': category,
            'needs_replacement': False, 'original': original,
            'replacement': None,
            'message': 'Это блюдо подходит для вашей цели — замена не нужна!'
        }

    category_mask = (df['Category'] == category) & (df['Food_Item'] != food_item)
    df_category = df[category_mask].reset_index(drop=True)

    if len(df_category) < 2:
        return {
            'food_item': food_item, 'goal': goal, 'category': category,
            'needs_replacement': True, 'original': original,
            'replacement': None,
            'message': f'В категории "{category}" недостаточно блюд для поиска замены'
        }

    # Сравниваем блюда-кандидаты по нутриционной плотности (на 100г) через StandardScaler
    scaler = StandardScaler()
    X_category = scaler.fit_transform(df_category[nutrition_features].astype(float))
    X_dish = scaler.transform(
        pd.DataFrame([dish[nutrition_features].astype(float)])[nutrition_features]
    )

    # Ранжируем ВСЕ блюда категории по расстоянию, а не только первые k —
    # если ближайшие несколько тоже "плохие" для цели, это не значит, что во
    # всей категории нет подходящей замены.
    knn = NearestNeighbors(n_neighbors=len(df_category))
    knn.fit(X_category)
    distances, indices = knn.kneighbors(X_dish)

    replacement_row = None
    for idx in indices[0]:
        candidate = df_category.iloc[idx]
        if int(candidate[target_col]) == 0:
            replacement_row = candidate
            break

    if replacement_row is None:
        return {
            'food_item': food_item, 'goal': goal, 'category': category,
            'needs_replacement': True, 'original': original,
            'replacement': None,
            'message': f'В категории "{category}" не нашлось подходящей замены под эту цель'
        }

    replacement = {f: float(replacement_row[f]) for f in nutrition_features}

    return {
        'food_item': food_item, 'goal': goal, 'category': category,
        'needs_replacement': True, 'original': original,
        'replacement': replacement_row['Food_Item'],
        'replacement_values': replacement,
        'message': f'Рекомендуем заменить на "{replacement_row["Food_Item"]}"'
    }

## 5. Тестирование на примерах

In [4]:
def print_result(result):
    """Красиво выводит результат поиска замены (все значения на 100г)."""
    if 'error' in result:
        print(f'Ошибка: {result["error"]}')
        return

    print(f'Блюдо:     {result["food_item"]} (на 100г)')
    print(f'Цель:      {result["goal"]}')
    print(f'Категория: {result["category"]}')
    print()

    if not result['needs_replacement']:
        print('Статус:  Замена не нужна — блюдо подходит для вашей цели!')
        print()
        print('КБЖУ на 100г:')
        for k, v in result['original'].items():
            print(f'  {k}: {v}')
    else:
        print('Статус:  Нужна замена')
        print()
        print('КБЖУ оригинала на 100г:')
        for k, v in result['original'].items():
            print(f'  {k}: {v}')
        print()
        if result['replacement']:
            print(f'Рекомендация:  {result["replacement"]}')
            print()
            print('КБЖУ замены на 100г:')
            for k, v in result['replacement_values'].items():
                print(f'  {k}: {v}')
        else:
            print(f'  {result["message"]}')
    print('=' * 55)

# Тест 1: Шоколадный торт при похудении
print('ТЕСТ 1:')
r1 = find_replacement('Шоколадный торт', 'Похудение')
print_result(r1)

# Тест 2: Борщ при наборе массы
print('ТЕСТ 2:')
r2 = find_replacement('Борщ', 'Набор массы')
print_result(r2)

ТЕСТ 1:
Блюдо:     Шоколадный торт (на 100г)
Цель:      Похудение
Категория: Десерт

Статус:  Нужна замена

КБЖУ оригинала на 100г:
  Calories (kcal): 385.0
  Protein (g): 5.5
  Carbohydrates (g): 52.5
  Fat (g): 18.5
  Fiber (g): 1.5
  Sugars (g): 38.5
  Sodium (mg): 180.0

Рекомендация:  Творожный пудинг

КБЖУ замены на 100г:
  Calories (kcal): 185.0
  Protein (g): 10.5
  Carbohydrates (g): 22.5
  Fat (g): 6.0
  Fiber (g): 0.3
  Sugars (g): 12.5
  Sodium (mg): 120.0
ТЕСТ 2:
Блюдо:     Борщ (на 100г)
Цель:      Набор массы
Категория: Суп

Статус:  Нужна замена

КБЖУ оригинала на 100г:
  Calories (kcal): 50.0
  Protein (g): 2.8
  Carbohydrates (g): 5.2
  Fat (g): 2.0
  Fiber (g): 1.2
  Sugars (g): 2.0
  Sodium (mg): 480.0

Рекомендация:  Суп с клёцками

КБЖУ замены на 100г:
  Calories (kcal): 82.0
  Protein (g): 3.5
  Carbohydrates (g): 11.5
  Fat (g): 2.5
  Fiber (g): 0.5
  Sugars (g): 0.8
  Sodium (mg): 420.0


## 6. Сводная таблица — несколько блюд и целей

In [5]:
test_cases = [
    ('Шоколадный торт',        'Похудение'),
    ('Медовик',                'Контроль сахара'),
    ('Борщ',                   'Набор массы'),
    ('Пельмени',               'Похудение'),
    ('Кефир 2.5%',             'Контроль сахара'),
    ('Гречневая каша на воде', 'Набор массы'),
    ('Сосиски варёные',        'Баланс/ЗОЖ'),
    ('Блины',                  'Похудение'),
    ('Яичница глазунья',       'Баланс/ЗОЖ'),
    ('Грецкий орех',           'Контроль сахара'),
]

print(f'{"Блюдо":<30} {"Цель":<20} {"Статус":<10} {"Замена"}')
print('-' * 95)

for food, goal in test_cases:
    result = find_replacement(food, goal)
    if 'error' in result:
        status = 'Ошибка'
        replacement = result['error']
    elif not result['needs_replacement']:
        status = 'OK'
        replacement = '— замена не нужна'
    elif result['replacement']:
        status = 'Замена'
        replacement = result['replacement']
    else:
        status = 'Замена невожможна'
        replacement = result['message'][:40]
    print(f'{food:<30} {goal:<20} {status:<10} {replacement}')

Блюдо                          Цель                 Статус     Замена
-----------------------------------------------------------------------------------------------
Шоколадный торт                Похудение            Замена     Творожный пудинг
Медовик                        Контроль сахара      Замена     Творожный пудинг
Борщ                           Набор массы          Замена     Суп с клёцками
Пельмени                       Похудение            Замена     Курица с картофелем запечённая
Кефир 2.5%                     Контроль сахара      OK         — замена не нужна
Гречневая каша на воде         Набор массы          OK         — замена не нужна
Сосиски варёные                Баланс/ЗОЖ           Замена     Котлеты мясные
Блины                          Похудение            OK         — замена не нужна
Яичница глазунья               Баланс/ЗОЖ           OK         — замена не нужна
Грецкий орех                   Контроль сахара      OK         — замена не нужна


## 7. Сохранение данных для финального пайплайна

In [6]:
knn_data = {
    'df':                df,
    'nutrition_features': nutrition_features,
    'label_to_target':   label_to_target
}

with open('../data/knn_data.pkl', 'wb') as f:
    pickle.dump(knn_data, f)


## 8. Выводы по Этапу 6

1. Реализован поиск замены блюда через алгоритм **k ближайших соседей (kNN)**.
2. Поиск ведётся среди блюд **той же категории** — суп заменяется супом, десерт десертом. Это обеспечивает логичность рекомендации.
3. Все значения КБЖУ используются **на 100г
4. Расстояние между блюдами считается в пространстве 7 нутриционных признаков после **масштабирования StandardScaler** 
5. Из блюд категории, ранжированных по расстоянию, выбирается ближайшее с меткой **target=0** - гарантированно подходящее под цель пользователя согласно правилам разметки.